# 13 — MCP: the wire format, then the SDK

**What you'll learn**

- That MCP is JSON-RPC with an agreed vocabulary — build the `initialize` handshake, `tools/list`, and `tools/call` by hand over an in-memory pipe, no SDK and no network
- The two failures a client must tell apart: a tool that runs and fails comes back as an ordinary result with `isError: true`, while an unknown method comes back as a JSON-RPC `error` object
- How to drive the official SDK: launch a `FastMCP` stdio server, then `initialize`, `list_tools`, `call_tool`, `read_resource`, `get_prompt` over a `ClientSession` — and reap the subprocess in the same cell
- Why the course pins `mcp>=1.28,<2` and teaches `FastMCP` (the whole agent ecosystem pins `mcp<2`)
- The payoff: adapt MCP tools into `shoplab.tools.Tool` and run a real triage on the chapter-02 `run_agent`, unchanged — same loop, remote transport

*Time: ~2 min on a first live run; under a minute cached. Cost: ~$0.02 (one live triage). Cached reruns are free.*

> **Before running this notebook:** `pip install -e ".[mcp]"` (once). It pulls in the official MCP Python SDK (`mcp>=1.28,<2`) — the `FastMCP` server and the stdio client we use in the back half. Everything else stays the same.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The triage at the end of this chapter still runs through `shoplab.llm.complete`, so Phoenix traces it like any other run — the only difference is that the tools it calls now live in another process. The handshakes and tool calls themselves are plain JSON-RPC, not model calls, so they do not show up as spans; watch the trace for the one triage turn. Optional as always: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Why a protocol at all

The chapter-02 agent hard-wired its tools as Python functions living in the same process as the loop. That is the right call when you own every tool and they are all small. It stops being the right call the moment the tools live somewhere else: a payments service another team maintains, a search index in a different language, a vendor's API you did not write and cannot import.

A protocol buys you *decoupling*. One agent talks to many tool servers — discovered, described, and swapped at runtime — without the loop knowing or caring how any of them is implemented. The Model Context Protocol (MCP) is that contract: a small, fixed way for a client (the agent's side) to ask a server (the tools' side) *what can you do* and *do this*. The rest of this chapter earns that one-paragraph claim by building the wire format by hand, then using the real SDK, then plugging an MCP server into the loop we already have.

## MCP is JSON-RPC with an agreed vocabulary

Strip the branding and MCP is [JSON-RPC 2.0](https://modelcontextprotocol.io/specification/2026-07-28) plus an agreed set of method names. A *request* is a small envelope: `jsonrpc`, an `id`, a `method` string, and `params`. A *response* carries the same `id` and either a `result` or an `error`. The vocabulary is the fixed list of methods — `initialize`, `tools/list`, `tools/call`, `resources/read`, `prompts/get` — and the shapes they expect. Nothing more.

To prove there is no magic under the SDK, we build a tiny server by hand over an in-memory function call: three envelope builders, a two-tool registry, and a dispatcher that switches on the method name. No socket, no subprocess — just the shapes.

In [ ]:
import json

def rpc(method, params=None, id=1):          # a JSON-RPC request envelope
    return {"jsonrpc": "2.0", "id": id, "method": method, "params": params or {}}

def ok(id, result):                          # a success response
    return {"jsonrpc": "2.0", "id": id, "result": result}

def err(id, code, message):                  # a protocol error response
    return {"jsonrpc": "2.0", "id": id, "error": {"code": code, "message": message}}

print(json.dumps(rpc("tools/list", id=1)))

In [ ]:
# The "server" state: two tools, one of which deliberately raises.
def _refund_preview(order_id, amount_usd):
    total = 391.50
    if amount_usd > total:
        raise ValueError(f"refund {amount_usd:.2f} exceeds order total {total:.2f}")
    return {"order_id": order_id, "amount_usd": amount_usd, "within_total": True}

TOOLS = {
    "get_order": {"schema": {"order_id": "string"},
                  "fn": lambda order_id: {"order_id": order_id, "status": "delivered",
                                          "total_usd": 391.50}},
    "refund_preview": {"schema": {"order_id": "string", "amount_usd": "number"},
                       "fn": _refund_preview},
}

def call_tool(id, params):
    tool = TOOLS.get(params["name"])
    if tool is None:                                     # unknown tool name: tools/call
        return ok(id, {"content": [{"type": "text",       # ran, found no such tool ->
                                    "text": f"Unknown tool: {params['name']}"}],
                       "isError": True})                  # isError result, as FastMCP does
    try:
        value = tool["fn"](**params.get("arguments", {}))
    except Exception as e:                               # the tool RAN and failed
        return ok(id, {"content": [{"type": "text", "text": str(e)}], "isError": True})
    return ok(id, {"content": [{"type": "text", "text": json.dumps(value)}],
                   "isError": False})

In [ ]:
def handle(req):                             # dispatch one request by method name
    m, p, i = req["method"], req.get("params", {}), req.get("id")
    if m == "initialize":
        return ok(i, {"protocolVersion": "2025-11-25",
                      "capabilities": {"tools": {"listChanged": False}},
                      "serverInfo": {"name": "toy-desk", "version": "0.1"}})
    if m == "tools/list":
        return ok(i, {"tools": [{"name": n, "inputSchema": t["schema"]}
                                for n, t in TOOLS.items()]})
    if m == "tools/call":
        return call_tool(i, p)
    return err(i, -32601, f"method not found: {m}")      # unknown method

## The initialize handshake

Every MCP session opens the same way, and the order is not optional. The client sends `initialize` announcing the protocol version it speaks and the capabilities it has; the server replies with *its* capabilities, its `serverInfo`, and the version it agreed to. The spec — [the lifecycle page, revision 2025-11-25](https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle) — is explicit that this initialization phase MUST be the first interaction between client and server. It is a capability handshake: both sides learn what the other supports before any real work.

In [ ]:
init_req = rpc("initialize", {
    "protocolVersion": "2025-11-25",
    "capabilities": {},                      # this toy client offers nothing extra
    "clientInfo": {"name": "toy-client", "version": "0.1"},
})
init_res = handle(init_req)["result"]
print("protocolVersion:", init_res["protocolVersion"])
print("serverInfo:     ", init_res["serverInfo"])
print("server caps:    ", init_res["capabilities"])

> **What you should see:** the server echoes a `protocolVersion`, returns `serverInfo` (a name and version), and advertises a `tools` capability. The two sides have agreed on a shared vocabulary before a single tool is named. That ordering is normative: [the lifecycle spec (2025-11-25)](https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle) requires the initialization phase to be the first interaction.

## tools/list, then tools/call

With the handshake done, discovery and use are two more methods. `tools/list` returns each tool's name and input schema — this is how a client learns what a server it has never seen before can do, at runtime, without reading its source. `tools/call` names one tool and its arguments and gets a result back. The result payload rides inside a `content` list (here, text), and every successful call carries an `isError` flag — which brings us to the interesting part.

In [ ]:
listed = handle(rpc("tools/list"))["result"]["tools"]
print("tools:", [t["name"] for t in listed])

got = handle(rpc("tools/call",
                 {"name": "get_order", "arguments": {"order_id": "ORD-7301"}}))["result"]
print("isError:", got["isError"])
print("content:", got["content"][0]["text"])

> **What you should see:** `tools/list` returns both tool names with their schemas, and `get_order` comes back with `isError: false` and the order payload as text inside `content`. A working call is a `result`, never an `error`.

## Two kinds of failure

A robust client has to tell apart two failures that look similar and are handled completely differently. The line the SDK actually draws is one question: did the request reach a handler at all?

`tools/call` is a method the server implements, so once a call is routed there, *whatever* goes wrong inside comes back as a `result` with `isError: true` — the tool raised on a bad argument, a business rule rejected the amount, or there is simply no tool by that name. The failure is data the model can read and recover from (re-ask, apologize, try another tool). What crosses the other line is naming a *method* the server does not implement at all — `frobnicate`, outside the MCP vocabulary. Then there is no handler to route to, no result to speak of, and the server returns a JSON-RPC `error` object. The first is a normal outcome of a working conversation; the second means the conversation itself broke.

In [ ]:
# 1) a tool that RAN and rejected the input -> isError result (not an error object)
over = handle(rpc("tools/call",
                  {"name": "refund_preview",
                   "arguments": {"order_id": "ORD-7301", "amount_usd": 999999.0}}))["result"]
print("tool ran+failed -> isError:", over["isError"], "|", over["content"][0]["text"])

# 2) an unknown TOOL NAME -> still an isError result: tools/call ran, found no such tool
unk = handle(rpc("tools/call", {"name": "no_such_tool", "arguments": {}}))["result"]
print("unknown tool    -> isError:", unk["isError"], "|", unk["content"][0]["text"])

# 3) an unknown METHOD the server does not implement -> a JSON-RPC error object
proto = handle(rpc("frobnicate", id=9))
print("bad method      -> has 'error':", "error" in proto, "|", proto["error"])

> **What to look for:** the over-limit `refund_preview` and the unknown *tool name* both come back as normal results with `isError: true` — in each case `tools/call` ran, and either the tool rejected the amount or there was no such tool — while the unknown *method* `frobnicate` returns a JSON-RPC `error` object with code `-32601`. The shipped server behaves the same way: call the live `refund_preview` over its limit, or ask for a tool that does not exist, and both are `isError` results; only an unmapped method is a protocol error. The tool-execution half is normative: [the tools spec](https://modelcontextprotocol.io/specification/2026-07-28/server/tools) states that tool-execution errors "are reported in tool results with `isError: true`", not as protocol errors.

One honest caveat about versions. Our live server (next section) negotiates revision `2025-11-25` — the newest revision that documents this `initialize` handshake, and the one `mcp` 1.x speaks. [The 2026-07-28 revision](https://modelcontextprotocol.io/specification/2026-07-28) reworked the lifecycle and made that handshake legacy. The vocabulary you built by hand is stable across both; the exact handshake framing is the 1.x one.

## From the toy to the SDK: FastMCP, and why `mcp` 1.x

The toy proved the shape. Nobody hand-rolls the framing in production — the official [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk) gives you `FastMCP`: decorate a function with `@mcp.tool()` and it derives the JSON schema from your type hints, publishes it under `tools/list`, and dispatches `tools/call` for you. `mcp_servers/shopdesk_server.py` is exactly that — it wraps the same `shoplab` world the whole course uses and re-publishes it over MCP:

- **Tools:** `get_order`, `get_customer`, `search_policy`, `check_inventory`, plus `refund_preview` — the one that raises, our live `isError` demo.
- **Resource:** `policy://{policy_id}`, a parameterised *template* the client reads by URI.
- **Prompt:** `triage_ticket`, a reusable message template parameterised by order and reason.

Why pin `mcp>=1.28,<2`? Because `pip install mcp` now installs the reworked 2.x line, which [renamed the high-level server class from `FastMCP` to `MCPServer`](https://py.sdk.modelcontextprotocol.io/whats-new/). The SDK's own README advises keeping a `<2` upper bound until you migrate, and the [maintained v1.x branch](https://github.com/modelcontextprotocol/python-sdk/tree/v1.x) is where `from mcp.server.fastmcp import FastMCP` still resolves. This is not just our preference: the agent ecosystem the course touches — arize-phoenix, agent-framework-core, crewai, strands-agents — all pin `mcp<2`, so a `<2` bound is what actually co-installs.

Now we speak to the real server. We launch it as a stdio subprocess; the `stdio_client` context manager *owns* that subprocess and reaps it on exit, so the server never outlives the cell. (This cell uses top-level `await`, which the notebook kernel supports.)

In [ ]:
import asyncio, sys
from pathlib import Path
from pydantic import AnyUrl
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.shared.exceptions import McpError

SERVER = Path("mcp_servers/shopdesk_server.py").resolve()

async def explore():
    params = StdioServerParameters(
        command=sys.executable, args=[str(SERVER)], env=os.environ.copy())
    async with stdio_client(params) as (read, write):        # owns + reaps the subprocess
        async with ClientSession(read, write) as session:
            init = await asyncio.wait_for(session.initialize(), timeout=10)
            print("handshake:", init.protocolVersion, "|", init.serverInfo.name)

            tools = await asyncio.wait_for(session.list_tools(), timeout=10)
            print("tools:", [t.name for t in tools.tools])

            r = await asyncio.wait_for(
                session.call_tool("get_order", {"order_id": "ORD-7301"}), timeout=10)
            print("get_order isError:", r.isError,
                  "status:", json.loads(r.content[0].text)["status"])

            rr = await asyncio.wait_for(
                session.read_resource(AnyUrl("policy://pol-returns")), timeout=10)
            print("resource policy://pol-returns:", rr.contents[0].text[:48], "...")

            gp = await asyncio.wait_for(
                session.get_prompt("triage_ticket",
                                   {"order_id": "ORD-7301", "reason": "boots pinch"}),
                timeout=10)
            print("prompt triage_ticket role:", gp.messages[0].role)

            over = await asyncio.wait_for(
                session.call_tool("refund_preview",
                                  {"order_id": "ORD-7301", "amount_usd": 999999.0}), timeout=10)
            print("refund_preview isError:", over.isError, "|", over.content[0].text)

            try:                                             # unknown resource -> protocol error
                await asyncio.wait_for(
                    session.read_resource(AnyUrl("policy://nope")), timeout=10)
            except McpError as e:
                print("protocol error (McpError):", e.error.message[:48], "...")
    print("teardown: stdio_client context exited; subprocess reaped.")

await explore()

> **What you should see:** the real handshake negotiates `2025-11-25`; `list_tools` returns the five tool names; `get_order` round-trips with `isError` false; the policy resource reads back as text; the triage prompt renders as a single `user` message; the over-limit `refund_preview` is `isError: true` (the same result-carried error as the toy); and reading an unknown policy raises `McpError` (the protocol-error path). One subtlety there: `isError` is a tools-only affordance, so a *resource* that fails has no in-band error channel and surfaces as a JSON-RPC error even though its handler did run — the opposite side of the line from `refund_preview`, which ran and failed yet stayed a result. The final line prints after the `async with` blocks unwind — proof the subprocess was reaped, not orphaned. FastMCP also logs a few `INFO` lines to stderr; those are the server narrating, not errors.

## The payoff: MCP tools into our chapter-02 loop

Here is the whole point of the chapter. `run_agent` from chapter 02 does not know or care where a tool runs — it calls `tool.fn(**args)` and reads back a JSON string. So if we can wrap each MCP tool in a `shoplab.tools.Tool` whose `fn` round-trips through a live session, the loop calls remote tools exactly as it called local ones. The loop code does not change. Only the transport does.

### The sync/async bridge

One wrinkle stands between us and that goal: `run_agent` is synchronous, but a `ClientSession` lives inside an async context that owns the subprocess. We bridge with a small runner that keeps the session alive on its own event loop in one background thread, and exposes a *sync* `call` that hands each coroutine to that loop with `run_coroutine_threadsafe(...).result(timeout)`. The context manager starts the thread on enter and, on exit, lets the async blocks unwind — which reaps the subprocess. Each MCP tool then becomes a `Tool` whose `fn` calls through the bridge.

In [ ]:
import threading

class MCPBridge:
    """Owns a background-thread event loop running one live MCP stdio session."""

    def __init__(self, server_path, python=sys.executable, timeout=10.0):
        self.params = StdioServerParameters(
            command=python, args=[str(server_path)], env=os.environ.copy())
        self.timeout = timeout
        self._loop = asyncio.new_event_loop()
        self._ready, self._stop = threading.Event(), threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._session, self.mcp_tools, self._exc = None, [], None

    def _run(self):
        asyncio.set_event_loop(self._loop)
        self._loop.run_until_complete(self._serve())

    async def _serve(self):
        try:
            async with stdio_client(self.params) as (read, write):
                async with ClientSession(read, write) as session:
                    await asyncio.wait_for(session.initialize(), timeout=self.timeout)
                    listed = await asyncio.wait_for(session.list_tools(), timeout=self.timeout)
                    self._session, self.mcp_tools = session, listed.tools
                    self._ready.set()
                    while not self._stop.is_set():          # keep the session alive
                        await asyncio.sleep(0.05)
        except Exception as e:                              # surface a startup failure
            self._exc = e
            self._ready.set()

    def __enter__(self):
        self._thread.start()
        if not self._ready.wait(timeout=self.timeout + 5):
            raise RuntimeError("MCP bridge did not become ready")
        if self._exc:
            raise self._exc
        return self

    def __exit__(self, *exc):
        self._stop.set()                                   # let _serve unwind the contexts
        self._thread.join(timeout=self.timeout + 5)        # subprocess reaped here
        self._loop.close()

    def _call_sync(self, name, args):
        fut = asyncio.run_coroutine_threadsafe(
            self._session.call_tool(name, args), self._loop)
        return fut.result(timeout=self.timeout)

    def _unwrap(self, res):
        """A CallToolResult -> a plain JSON-able value run_tool can serialise."""
        if res.structuredContent is not None:
            sc = res.structuredContent
            if isinstance(sc, dict) and set(sc) == {"result"}:   # FastMCP list wrap
                return sc["result"]
            return sc
        if res.content and getattr(res.content[0], "text", None) is not None:
            txt = res.content[0].text
            if res.isError:
                return {"error": txt}
            try:
                return json.loads(txt)
            except ValueError:
                return txt
        return {"error": "empty MCP result"} if res.isError else None

    def as_tools(self):
        """dict[name -> shoplab Tool] whose fn round-trips through the session."""
        from shoplab.tools import Tool
        tools = {}
        for mt in self.mcp_tools:
            def make(n):
                def fn(**args):
                    return self._unwrap(self._call_sync(n, args))
                return fn
            tools[mt.name] = Tool(name=mt.name, description=mt.description or "",
                                  params=mt.inputSchema, fn=make(mt.name))
        return tools

In [ ]:
from shoplab.tools import standard_tools
from shoplab.loop import run_agent

with MCPBridge(SERVER) as bridge:
    # The MCP server has no terminator, so add our local finish tool.
    tools = {**bridge.as_tools(), "finish": standard_tools()["finish"]}
    print("tools now sourced from MCP:", [n for n in tools if n != "finish"])

    task = ("Triage this Larkspur return. Order ORD-7312. Customer's reason: 'I opened "
            "the Torrent boots, wore them one evening indoors, they pinch at the toes. "
            "Repacked with tags. Please refund my original payment.' Look up the order "
            "and the customer, search policy for the governing rule, then call finish "
            "with decision, policy_id, and refund_usd.")
    system = ("You are the Larkspur Outfitters ops desk. Gather the order, the customer, "
              "and ONE policy search, then immediately call finish with decision, "
              "policy_id, refund_usd. Do not repeat lookups.")

    calls = []
    result = run_agent(task, tools, system=system, max_steps=8,
                       on_step=lambda step, msg: calls.append(
                           [tc.function.name for tc in (msg.tool_calls or [])]))
    print("trajectory:", calls)
    print("stop_reason:", result.stop_reason, "| decision:", result.answer)

print("bridge closed; subprocess reaped in the bridge thread.")

> **What you should see:** the toolset the loop now draws from is the MCP server's five tools (plus the local `finish`); the agent gathers the order and the customer, searches policy, and ends at `finish` with a structured decision — a refund minus a restocking fee, i.e. a `partial_refund`, the shape an opened, in-window return from a non-VIP customer takes.

Two honest notes on the numbers. Temperature 0 is not determinism: the exact trajectory, the policy id the model cites, and the fee it computes wander run to run — the gold answer `rules.decide` fixed in chapter 04 is `pol-restocking` for `$170.99`, and a small live model may land a few dollars off or cite a neighbouring policy (this run did). Grading that gap was the job of chapters 04 and 06; the invariant *here* is narrower and more important — `run_agent` reached a structured decision with every tool living in another process. The loop is the byte-for-byte chapter-02 loop; only the tools' transport changed, and the bridge's `__exit__` reaps the subprocess.

## Resources, tools, prompts — three primitives

MCP is not only tools. The [2026-07-28 specification](https://modelcontextprotocol.io/specification/2026-07-28) frames three primitives, each with a different *controller* — who decides when it is used:

| Primitive | What it carries | Who decides to use it | In `shopdesk_server.py` |
|---|---|---|---|
| Tool | a function the model may call (computation, side effects) | the model, mid-loop | `get_order`, `refund_preview`, ... |
| Resource | context and data the client reads by URI | the application | `policy://{policy_id}` template |
| Prompt | a templated message or workflow | the user | `triage_ticket` |

The distinction is not academic. A refund is a *tool* because the model chooses to call it while reasoning; a policy document is a *resource* because the app decides to pull it into context; a triage template is a *prompt* because a person invokes it to start a workflow. Modelling stock as a resource the app fetches versus a tool the model calls is a real design choice, and it changes who is in control.

## Recap

| Concept | One-liner |
|---|---|
| MCP = JSON-RPC + vocabulary | requests carry `method`/`params`, responses carry `result` or `error`; the vocabulary is `initialize`, `tools/list`, `tools/call`, `resources/read`, `prompts/get`. |
| `initialize` handshake | the required first interaction: both sides exchange capabilities and agree a protocol version before any work. |
| `isError` vs protocol error | a tool that ran and failed rides back as a `result` with `isError: true`; an unknown method returns a JSON-RPC `error` object. |
| `FastMCP` | the 1.x SDK server: `@mcp.tool()` derives a schema from type hints and dispatches the wire methods for you. |
| The `mcp<2` pin | `pip install mcp` now installs 2.x (which renamed `FastMCP` to `MCPServer`); pin `mcp>=1.28,<2`, as the ecosystem does. |
| stdio transport + teardown | `stdio_client` launches the server as a subprocess and reaps it when its context exits — no orphan. |
| The adapter | wrap each MCP tool in a `shoplab.tools.Tool`; `run_agent` calls remote tools exactly as local ones — loop unchanged, transport changed. |
| Three primitives | tools (model-controlled), resources (app-controlled), prompts (user-controlled). |

## Exercises

1. **Add a tool over the wire.** Give `mcp_servers/shopdesk_server.py` a new `@mcp.tool()` (say, a `get_reviews(sku)` lookup, or a `days_left_in_window` helper), relaunch the client cell, and confirm it appears in `list_tools` and that the adapted `run_agent` can call it — without touching the loop. What, exactly, did you have to change, and what did you not?
2. **Draw the failure line yourself.** The toy now answers an unknown *tool name* with an `isError` result (the `tools/call` method ran; there was simply no such tool) and keeps the JSON-RPC `error` for an unknown *method* like `frobnicate`. Confirm the tool-name half against the live server — `session.call_tool("no_such_tool", {})` comes back `isError: true`, matching the toy — then state in one sentence the rule that draws the line: is it "the tool failed" or "the request never reached a handler"? Where does a wrong-typed argument fall, and why?
3. **Expose stock as a resource, not a tool.** Add an `inventory://{sku}` resource to the server that returns the stock line, read it with `read_resource`, and compare it to calling `check_inventory` as a tool. When does stock belong as a resource the app pulls into context versus a tool the model chooses to call? Tie your answer back to the three-primitives table.

**Next up:** chapter 14 — agents talking to agents. MCP decoupled the agent from its tools; A2A takes the same move one layer up, so one agent can delegate to another over the network.